In [ ]:
# =========================
# Noah Russell 
# Groq LLM demo  


!pip -q install groq

import os
import json
import re
from getpass import getpass
from groq import Groq

# -------------------------
# API key
# (In Colab, this keeps it from showing up in your exported HTML/PDF.)
# -------------------------
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key (input hidden): ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

def stream_chat(messages, temperature=1.0, max_tokens=600):
    """Streams assistant output (so you can show you actually used stream=True)."""
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_tokens,
        top_p=1,
        reasoning_effort="medium",
        stream=True,
    )
    text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
            text += delta
    print()  # newline
    return text

def nonstream_chat(messages, temperature=0.7, max_tokens=600):
    """Non-streaming call is easier when we need to parse JSON reliably."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_tokens,
        top_p=1,
        reasoning_effort="medium",
        stream=False,
    )
    return resp.choices[0].message.content

def extract_json_object(text):
    """
    Tries to pull a JSON object out of the model output.
    (Models sometimes add extra text or wrap JSON in code fences.)
    """
    cleaned = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE).replace("```", "").strip()

    # try direct parse first
    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # otherwise try grabbing the first {...} block
    m = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None

# 10a) Multi-round convo that needs memory + logic
print("\n==============================")
print("10a) Multi-round memory + logic task (3 rounds)")
print("==============================\n")

SYSTEM = {
    "role": "system",
    "content": (
        "You are a helpful assistant. "
        "When asked for calculations, be careful and keep the math consistent."
    )
}

history = [SYSTEM]

# Round 1
history.append({
    "role": "user",
    "content": (
        "Round 1: Remember this shopping plan.\n"
        "Budget: $20.\n"
        "Items:\n"
        "- apples: $4\n"
        "- bread: $3\n"
        "- milk: $5\n"
        "Just reply 'Got it' and confirm the budget."
    )
})

print("Assistant (Round 1): ", end="")
r1 = stream_chat(history, temperature=0.8)
history.append({"role": "assistant", "content": r1})

# Round 2
history.append({
    "role": "user",
    "content": (
        "Round 2: Add these items and remember them too:\n"
        "- eggs: $6\n"
        "- candy: $2\n"
        "Reply 'Added' and restate the full item list."
    )
})

print("Assistant (Round 2): ", end="")
r2 = stream_chat(history, temperature=0.8)
history.append({"role": "assistant", "content": r2})

# Round 3: ask for strict JSON
history.append({
    "role": "user",
    "content": (
        "Round 3: Now compute the total cost and how much money is left.\n"
        "Return ONLY valid JSON like this:\n"
        '{ "budget": 20, "total": 0, "remaining": 0, "items": [{"name":"", "price":0}], "notes": "" }\n'
        "No extra text."
    )
})

print("Assistant (Round 3, correct context): ", end="")
r3 = nonstream_chat(history, temperature=0.7)
print(r3)

# Python ground truth check (so we can prove correctness)
prices = {"apples": 4, "bread": 3, "milk": 5, "eggs": 6, "candy": 2}
budget = 20
true_total = sum(prices.values())
true_remaining = budget - true_total

print("\nPython check:")
print("True total =", true_total, "| True remaining =", true_remaining)

# 10b) Mistake demo(s) + why
print("\n==============================")
print("10b) Mistake demos + why they happen")
print("==============================\n")

# (1) Memory mistake: DON'T send the earlier history
bad_messages = [
    SYSTEM,
    {
        "role": "user",
        "content": (
            "Compute the total cost and remaining budget from our earlier shopping list.\n"
            "Return JSON with budget, total, remaining."
        )
    }
]

print("Assistant (BAD call, missing memory): ", end="")
bad = stream_chat(bad_messages, temperature=0.9)

print("\nWhy this is a mistake case (memory):")
print("- The API call is stateless: the model only sees what you send in 'messages'.")
print("- If you don’t include the earlier rounds, it literally can’t know the list/budget.")
print("- Best case: it asks for the missing info (like it did). Worst case: it guesses numbers.")

# (2) Logic mistake: sometimes the model messes up arithmetic (especially with randomness)
print("\nAssistant (Extra mistake, possible math slip): ", end="")
logic_test = [
    {"role": "system", "content": "Return ONLY JSON. Do the math quickly."},
    {"role": "user", "content": 'Budget=20. Items: apples 4, bread 3, milk 5, eggs 6, candy 2. Return {"budget":20,"total":?,"remaining":?}'}
]
logic_out = nonstream_chat(logic_test, temperature=1.2)
print(logic_out)

print("\nWhy this can be wrong (logic):")
print("- LLMs aren’t calculators. They can do math, but they can also slip.")
print("- Higher temperature (more randomness) can make mistakes more likely.")
print("- That’s why you don’t blindly trust numbers from a model without checking.")

# 10c) Fix: keep history + validate + (optionally) force correction
print("\n==============================")
print("10c) Fix: keep history + validate output in code")
print("==============================\n")

obj = extract_json_object(r3)

needs_retry = False
if obj is None:
    print("Model didn’t give clean JSON. That’s fixable: tighten the prompt and retry.")
    needs_retry = True
else:
    model_total = obj.get("total", None)
    model_remaining = obj.get("remaining", None)

    if model_total != true_total or model_remaining != true_remaining:
        print("Model JSON didn’t match Python’s totals. We should force a correction.")
        needs_retry = True
    else:
        print("Model JSON matched Python verification. Looks correct.")
        print("Still, the safe approach is: keep the full message history + validate the math in code.")

if needs_retry:
    fix_prompt = {
        "role": "user",
        "content": (
            "Your JSON was missing/incorrect. Recalculate using:\n"
            f"Budget = {budget}\n"
            f"Items/prices = {prices}\n"
            f"Total must equal {true_total} and remaining must equal {true_remaining}.\n"
            "Return ONLY valid JSON. No extra text."
        )
    }

    fixed_history = history + [{"role": "assistant", "content": r3}] + [fix_prompt]
    fixed = nonstream_chat(fixed_history, temperature=0.2)

    print("\nAssistant (Corrected JSON):")
    print(fixed)

print("\nDONE: This covers 10a (multi-round memory+logic), 10b (mistakes + why), 10c (fix).")



10a) Multi-round memory + logic task (3 rounds)

Assistant (Round 1): Got it. Budget: $20.
Assistant (Round 2): Added.  
Full item list:  
- apples: $4  
- bread: $3  
- milk: $5  
- eggs: $6  
- candy: $2
Assistant (Round 3, correct context): {
  "budget": 20,
  "total": 20,
  "remaining": 0,
  "items": [
    {"name": "apples", "price": 4},
    {"name": "bread", "price": 3},
    {"name": "milk", "price": 5},
    {"name": "eggs", "price": 6},
    {"name": "candy", "price": 2}
  ],
  "notes": ""
}

Python check:
True total = 20 | True remaining = 0

10b) Mistake demos + why they happen

Assistant (BAD call, missing memory): I don’t have the details of the shopping list or the budget you’re referring to. Could you please provide the list of items with their costs (or the total amount you’ve spent so far) and the total budget you have allocated? Once I have that information, I can calculate the total cost and the remaining budget and return the result in JSON format.

Why this is a mista